In [3]:
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from scikeras.wrappers import KerasClassifier
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

with open('proccessed_data.pkl', 'rb') as file:
    data = pickle.load(file)
X = data.drop('Married', axis=1)
Y = data['Married']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_test = scalar.transform(X_test)
opt = tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss = tensorflow.keras.losses.BinaryCrossentropy()
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 31 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Age                               10000 non-null  int64  
 1   Gender                            10000 non-null  int64  
 2   Partner_Age                       10000 non-null  int64  
 3   Partner_Gender                    10000 non-null  int64  
 4   Years_Together                    10000 non-null  float64
 5   Age_Difference                    10000 non-null  int64  
 6   Intercaste                        10000 non-null  int64  
 7   Are_They_Happy                    10000 non-null  int64  
 8   Want_To_Marry                     10000 non-null  int64  
 9   Compatibility_Score               10000 non-null  int64  
 10  Married                           10000 non-null  int64  
 11  Religion_Buddhist                 10000 non-null  float64
 12  Religion_Christi

In [4]:
model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


c:\Users\rajes\miniconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [5]:
import datetime


log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [6]:
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(
    X_train,
    Y_train,
    validation_data=(X_test, Y_test),
    epochs=50,
    batch_size=10,
    callbacks=[early_stopping_callback, tensorboard_callback]
)

Epoch 1/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - accuracy: 0.9057 - loss: 0.2420 - val_accuracy: 0.9325 - val_loss: 0.1640
Epoch 2/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9435 - loss: 0.1491 - val_accuracy: 0.9420 - val_loss: 0.1514
Epoch 3/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.9467 - loss: 0.1354 - val_accuracy: 0.9400 - val_loss: 0.1442
Epoch 4/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9496 - loss: 0.1264 - val_accuracy: 0.9390 - val_loss: 0.1464
Epoch 5/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9534 - loss: 0.1184 - val_accuracy: 0.9395 - val_loss: 0.1449
Epoch 6/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9561 - loss: 0.1091 - val_accuracy: 0.9455 - val_loss: 0.1314
Epoch 7/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9600 - loss: 0.1004 - val_accuracy: 0.9415 - val_loss: 0.1288
Epoch 8/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.9613 - loss: 0.0948 - val_accuracy:

In [7]:
model.save('marriage_model.h5')

In [8]:
%load_ext tensorboard
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 10432), started 1 day, 0:14:13 ago. (Use '!kill 10432' to kill it.)